# External Memory Retriever

This example shows a small adapter pattern for using an external memory service as a LlamaIndex retriever. The service client is intentionally in-memory here, but the same shape can wrap an HTTP API, database query, or any other persistent memory backend.

The key idea is to keep service-specific code inside the client and have the retriever return standard `NodeWithScore` objects.

In [ ]:
from dataclasses import dataclass
from typing import Any, Dict, List, Protocol

from llama_index.core.retrievers import BaseRetriever
from llama_index.core.schema import NodeWithScore, QueryBundle, TextNode

## Define the memory client boundary

A real implementation could call `requests.get(...)`, `httpx.Client.post(...)`, or an SDK method inside `search`. Keeping that logic behind a tiny client interface makes the LlamaIndex retriever easy to test.

In [ ]:
@dataclass
class MemoryHit:
    text: str
    score: float
    metadata: Dict[str, Any]


class MemoryClient(Protocol):
    def search(self, query: str, limit: int) -> List[MemoryHit]: ...


class DemoMemoryClient:
    def __init__(self, memories: List[MemoryHit]) -> None:
        self.memories = memories

    def search(self, query: str, limit: int) -> List[MemoryHit]:
        terms = query.lower().split()
        matches = [
            memory
            for memory in self.memories
            if any(term in memory.text.lower() for term in terms)
        ]
        return sorted(matches, key=lambda memory: memory.score, reverse=True)[:limit]

## Wrap the client as a retriever

Subclass `BaseRetriever` and implement `_retrieve`. Convert each service result into a `TextNode`, preserving metadata that may help downstream synthesis or citation.

In [ ]:
class ExternalMemoryRetriever(BaseRetriever):
    def __init__(self, client: MemoryClient, similarity_top_k: int = 3) -> None:
        super().__init__()
        self.client = client
        self.similarity_top_k = similarity_top_k

    def _retrieve(self, query_bundle: QueryBundle) -> List[NodeWithScore]:
        hits = self.client.search(
            query=query_bundle.query_str,
            limit=self.similarity_top_k,
        )
        return [
            NodeWithScore(
                node=TextNode(text=hit.text, metadata=hit.metadata),
                score=hit.score,
            )
            for hit in hits
        ]

## Retrieve memory before generation

The retriever can now be passed anywhere LlamaIndex expects a retriever, or called directly before merging memory snippets with regular RAG context.

In [ ]:
client = DemoMemoryClient(
    memories=[
        MemoryHit(
            text="The user prefers concise answers with source links.",
            score=0.92,
            metadata={"memory_id": "pref-1", "kind": "preference"},
        ),
        MemoryHit(
            text="The user is evaluating retriever patterns for a RAG app.",
            score=0.87,
            metadata={"memory_id": "proj-7", "kind": "project"},
        ),
    ]
)

retriever = ExternalMemoryRetriever(client, similarity_top_k=2)
nodes = retriever.retrieve("What retriever pattern should I use?")

for node in nodes:
    print(node.score, node.node.text, node.node.metadata)

Because the adapter returns normal `NodeWithScore` objects, the rest of the application does not need to know whether the context came from a vector index, a memory service, or both.